This notebook requires an stl in the [asset directory](assets). The stl is added into MuJoCo's spec with a simple box collision geometry.

In [ ]:
import mujoco
import mujoco.viewer
import trimesh
import time

In [ ]:
mesh_dir = "assets"
mesh_file = "00497770_A.stl"

mesh_scale = 1e-3
mesh = trimesh.load_mesh(rf"{mesh_dir}/{mesh_file}")
mesh_center = mesh.bounding_box.centroid
mesh_extent = mesh.bounding_box.extents

pos_adjust = -mesh_center * mesh_scale
collision_box_half_sz = mesh_extent * mesh_scale / 2.0

px, py, pz = pos_adjust
sx, sy, sz = collision_box_half_sz

world_spec = mujoco.MjSpec.from_string(
    f"""
<mujoco model="ur5e_dual_scene">

    <compiler angle="radian" meshdir="{mesh_dir}"/>

    <option
        gravity="0 0 -9.81"
        timestep="0.001"
        integrator="implicitfast"
        iterations="100"/>

    <asset>
        <mesh
            name="my_part_mesh"
            file="{mesh_file}"
            scale="{mesh_scale} {mesh_scale} {mesh_scale}"/>
    </asset>

    <worldbody>
        <light
            diffuse=".5 .5 .5"
            pos="0 0 3"
            dir="0 0 -1"/>

        <geom
            name="floor"
            type="plane"
            size="2 2 0.1"
            rgba=".9 .9 .9 1"
            friction="1 0.01 0.001"/>

        <!-- Here, we want it slightly above floor to see it drop -->
        <body name="my_part" pos="0 0 0.3">
            <freejoint/>

            <!-- Collision box -->
            <!-- size: half of extent of the STL mesh * STL_scale -->
            <!-- other configs: density? -->
            <geom
                name="my_part_collision"
                type="box"
                size="{sx} {sy} {sz}"
                mass="0.1"
                friction="1 0.01 0.001"
                rgba="0 1 0 0.3"/> <!-- green -->

            <!-- Visual-only STL -->
            <!-- visual_geom_position = -(STL_center * STL_scale) -->
            <!-- other configs: solref, solimp -->
            <geom
                name="my_part_visual"
                type="mesh"
                mesh="my_part_mesh"
                pos="{px} {py} {pz}"
                contype="0"
                conaffinity="0"
                mass="0"
                rgba="0.2 0.5 0.9 0.7"/> <!-- blue -->
        </body>
    </worldbody>
</mujoco>
"""
)


In [ ]:
model = world_spec.compile()
data = mujoco.MjData(model)

In [ ]:

body_id = mujoco.mj_name2id(
    model,
    mujoco.mjtObj.mjOBJ_BODY,
    "my_part",
)

with mujoco.viewer.launch_passive(model, data) as viewer:
    step_count = 0

    while viewer.is_running():
        wall_start = time.perf_counter()

        mujoco.mj_step(model, data)
        step_count += 1

        if step_count % 100 == 0:
            position = data.xpos[body_id].copy()
            angular_velocity = data.cvel[body_id, 0:3].copy()
            linear_velocity = data.cvel[body_id, 3:6].copy()

            print(f"time:     {data.time:.3f} s")
            print(f"position: {position}")
            print(f"linear:   {linear_velocity}")
            print(f"angular:  {angular_velocity}")
            print(f"contacts: {data.ncon}")
            print()

        viewer.sync()

        sleep_time = model.opt.timestep - (
            time.perf_counter() - wall_start
        )

        if sleep_time > 0:
            time.sleep(sleep_time)

# Alternative

Correct the translation before import. The same world_spec can be used.

In [ ]:
mesh_scale = 1e-3
mesh_dir = "assets"
mesh_file = "00497770_A.stl"
mesh_file_centered = "00497770_A_centered.stl"

mesh = trimesh.load_mesh(rf"{mesh_dir}/{mesh_file}")
mesh.apply_translation(-mesh.bounding_box.centroid)
mesh.export(rf"{mesh_dir}/{mesh_file_centered}")

# use new mesh file in mujoco
mesh_file = mesh_file_centered

mesh = trimesh.load_mesh(rf"{mesh_dir}/{mesh_file}")
mesh_center = mesh.bounding_box.centroid
mesh_extent = mesh.bounding_box.extents

collision_box_half_sz = mesh_extent * mesh_scale / 2.0

px, py, pz = [0,0,0]
sx, sy, sz = collision_box_half_sz